[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/data-engineering-from-scratch-to-advanced/blob/main/notebooks/topics/20_system-design.ipynb)

*Data Engineering from Scratch to Advanced · Notebook 20 of 21*

# Data System Design

How senior engineers think: a repeatable design framework, sizing, choosing tools and walkthroughs of clickstream analytics, CDC to a lake, real-time dashboards and data platforms, with the trade-offs interviewers want to hear.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** Nobody wants the one right answer. They want to watch you ask questions, estimate, choose, and name what could go wrong.

**What interviewers really test here:** A clear structure, sizing, trade-offs and failure handling. This often decides senior offers.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. How to approach any design question *(Basics)*
- [ ] 2. Walkthrough: clickstream analytics *(Core)*
- [ ] 3. Walkthrough: replicate a database to the lake *(Core)*
- [ ] 4. Scale, cost and the human side *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline
import math
import numpy as np
import matplotlib.pyplot as plt
PLAIN = True

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Design questions to practise aloud

| Prompt | Must mention |
| --- | --- |
| Clickstream analytics | collector, Kafka, dedupe, sessions, late data |
| Replicate a database | CDC, ordering, merge, schema changes |
| Ride-share live metrics | streaming windows, real-time OLAP, surge by area |
| Payments ledger reporting | exactly-once, reconciliation, audit, immutability |
| IoT sensors at huge scale | partitioning, downsampling, retention tiers |
| Feature store | online and offline, point-in-time correctness |
| Company data platform | lakehouse, catalog, access control, cost, ownership |
| Search or RAG ingestion | incremental updates, deletes, freshness |

---
# Part 2: Lessons, step by step

## Lesson 1: How to approach any design question

*Level: Basics*

"Design a pipeline for clickstream data." Nothing is specified, and that's the test. Senior engineers ask questions, estimate, and explain trade-offs, and they don't jump straight to naming tools.



| Step | Ask yourself | Time (of 45 min) |
| --- | --- | --- |
| 1. Requirements | Who uses it? Which questions? How fresh? How correct? | 5–8 |
| 2. Size it | Events per second? GB per day? Retention? Peak? | 3–5 |
| 3. Data model | What's the grain? Keys? Partitions? | 5 |
| 4. Architecture | Ingest, store, process, serve | 15 |
| 5. Failure | Reruns? Late data? Duplicates? Bad data? | 5–8 |
| 6. Operate | Monitoring, cost, security, growth | 5 |



**Sizing, worked.** 10 million users, 50 events each, 1 KB per event, peak 3×:



- 500 million events a day ÷ 86,400 ≈ **5,787 per second**, with a peak of about **17,400**.
- 500 million × 1 KB = **500 GB a day**.
- Raw kept 30 days: 15 TB. Clean Parquet, 5× smaller, kept a year: 36.5 TB.
- Peak of 17 MB/s, doubled for headroom, at 10 MB/s per partition: **4 Kafka partitions**.
- A 5 TB batch in a 2-hour window at 20 MB/s per core and 70% efficiency: **50 cores**.



**Choosing based on need:**



| Need | Choice |
| --- | --- |
| Freshness in seconds | streaming |
| Minutes | micro-batch |
| Hours or a day | batch (simplest) |
| Serve by key in under 50 ms | key-value store (DynamoDB, Redis) |
| Fast aggregates under 1 s | real-time OLAP (Pinot, Druid, ClickHouse) |
| Everything else | warehouse |



**Say the trade-off out loud:** "I'd choose hourly batch rather than streaming, because the dashboard's read daily, and it's far simpler to run. If that changes, we can move to streaming from the same Kafka topic."

**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q2](#q2), [Q3](#q3), [Q10](#q10), [Q4](#q4), [Q11](#q11), [Q5](#q5). Try one now, or carry on and do them in Part 3.

## Lesson 2: Walkthrough: clickstream analytics

*Level: Core*

**Requirements:** a website wants daily active users, funnels and an hourly traffic view. 500 GB a day, and some phones send events late.



```text
Browser → Collector API → Kafka → (a) raw to S3, partitioned by hour
                                  (b) Spark: clean, dedupe, sessionise
                                       → Iceberg silver table
                                       → dbt marts → Warehouse → BI
                 Airflow schedules batch; alerts on freshness and volume
```



**Why these choices?**



| Choice | Why |
| --- | --- |
| A collector API | validates events, and keeps the app separate from Kafka |
| Kafka | absorbs spikes, and lets many consumers read |
| Keep raw | so bugs can be fixed by reprocessing |
| Partition by event date | queries filter by day; late data updates old days |



**Grain:** one row per event, with a unique event\_id. Then one row per session, and one per user per day.



**Failure cases to raise:**



- **Duplicates:** dedupe on event\_id. DAU uses distinct users, so duplicates don't inflate it: events a, a, b on d1 give 2.
- **Bots:** filter them out, or they'll distort everything. A user with 3 events against a limit of 2 is dropped here, and real rules are subtler.
- **Late events:** reprocess the last 3 days each run, overwriting partitions.
- **Outage:** with 7 days of Kafka retention, a 6-hour outage can replay. At 3× speed, it catches up in 3 hours.



**Real-time view:** a Flink or Spark job writes per-minute counts, and hourly rollups make dashboards fast: minutes 0 and 59 roll into hour 0.

**✏️ Your turn:** you now have what you need for [Q12](#q12), [Q6](#q6), [Q14](#q14), [Q16](#q16), [Q7](#q7). Try one now, or carry on and do them in Part 3.

## Lesson 3: Walkthrough: replicate a database to the lake

*Level: Core*

**Requirements:** copy production PostgreSQL tables to the lake within 15 minutes, including deletes, with no load on the app.



```text
Postgres → WAL → Debezium → Kafka → Spark micro-batch → MERGE into Iceberg → dbt
```



| Decision | Reason |
| --- | --- |
| CDC instead of a daily dump | sees deletes, fresher, light on the source |
| Primary key as Kafka key | keeps each row's changes in order |
| Merge-on-read | frequent updates; compact in the background |
| Initial snapshot, then CDC | the log doesn't hold all of history |



**Compact before merging.** Keep the highest sequence per id. Out-of-order events (seq 2: B, seq 1: OLD) must end at B.



**Hard parts to mention:**



- **Schema changes:** a new column must flow through, so use a schema registry and agreed rules.
- **Backfills:** a new table needs a snapshot, and then a switch to the stream.
- **Replication slots:** if the consumer stops, the database keeps its log and can fill its disk. Monitor this.
- **Reconciliation:** compare counts and sums between source and lake each day.

**✏️ Your turn:** you now have what you need for [Q13](#q13), [Q18](#q18). Try one now, or carry on and do them in Part 3.

## Lesson 4: Scale, cost and the human side

*Level: Advanced*

**Partition layout.** Choose the coarsest grain under your size limit: 10 GB a day with a 500 GB limit suits monthly partitions; 500 GB a day with a 100 GB limit suits days, and 5 TB a day needs hours.



**Serving at scale.** Reads of 100,000 per second at 10,000 per shard, writes of 10,000 at 2,000 per shard, and 2 TB at 500 GB per shard: 10, 5 and 4 shards are needed, so **10**.



**Noisy neighbours.** One customer shouldn't starve others. With demands 10, 50 and 100 and capacity 90, max-min fairness gives 10, 40 and 40.



**Estimate cost in the design.** 100 TB at 23, 300 hours × 10 nodes × 0.5, and 50 MB/s of streaming comes to 5,260 a month, with storage the largest, at these made-up prices. Would batch be cheaper? Say so.



**Will it last?** Load of 100, capacity 400, growing 10% monthly: 15 months. Know when your design breaks.



**Rank your risks** by likelihood × impact. Schema change (4 × 4 = 16) comes before late data (10) and source outage (10). Say what you'd do about the top ones.



**Data mesh** gives domain teams ownership of their data as a product, with a central platform team. It solves a central team becoming a bottleneck, and needs mature teams. Datasets with no owner, or one team owning everything, are warning signs.



**What separates the very best?**



- They ask what decision the data supports.
- They pick the simplest thing that works, and say what would change that.
- They design for failure, and measure cost.
- They explain to non-engineers clearly.

**✏️ Your turn:** you now have what you need for [Q17](#q17), [Q8](#q8), [Q19](#q19), [Q15](#q15), [Q20](#q20), [Q9](#q9). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

<a id="q1"></a>
### Q1. Size the Ingest

🟢 Warm-up · *Start with numbers*

From daily users and events per user, return (average events per second, peak at the multiple, GB per day at size bytes), rounded to 1 decimal.

**Examples**

- `users = 10000000, per_user = 50, peak = 3, size = 1000` → `(5787.0, 17361.1, 500.0)`

In [ ]:
def size_ingest(users, per_user, peak, size):
    """
    From daily users and events per user, return (average events per second,
    peak at the multiple, GB per day at size bytes), rounded to 1 decimal.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(size_ingest, [
    ((10000000, 50, 3, 1000), (5787.0, 17361.1, 500.0)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Step by step.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Compact.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Step by step.

```python
def size_ingest(users, per_user, peak, size):
    per_day = users * per_user
    avg = per_day / 86400
    gb = per_day * size / 1e9
    return (round(avg, 1), round(avg * peak, 1), round(gb, 1))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Compact.

```python
def size_ingest(users, per_user, peak, size):
    d = users * per_user
    return (round(d / 86400, 1), round(d / 86400 * peak, 1), round(d * size / 1e9, 1))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Step by step. That costs O(1) time and O(1) space."

"I can do better: Compact. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/system-design/size-the-ingest)

<a id="q2"></a>
### Q2. Storage Over Time

🟢 Warm-up · *Layers and retention*

Given raw GB per day, Parquet compression ratio and retention days for raw and clean, return TB for each, rounded to 2 decimals.

**Examples**

- `gb_day = 500, ratio = 5, raw_days = 30, clean_days = 365` → `(15.0, 36.5)`

In [ ]:
def storage_plan(gb_day, ratio, raw_days, clean_days):
    """
    Given raw GB per day, Parquet compression ratio and retention days for raw
    and clean, return TB for each, rounded to 2 decimals.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(storage_plan, [
    ((500, 5, 30, 365), (15.0, 36.5)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Step by step.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Compact.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Step by step.

```python
def storage_plan(gb_day, ratio, raw_days, clean_days):
    raw = gb_day * raw_days / 1000
    clean = gb_day / ratio * clean_days / 1000
    return (round(raw, 2), round(clean, 2))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Compact.

```python
def storage_plan(gb_day, ratio, raw_days, clean_days):
    return (round(gb_day * raw_days / 1e3, 2), round(gb_day * clean_days / ratio / 1e3, 2))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Step by step. That costs O(1) time and O(1) space."

"I can do better: Compact. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/system-design/storage-over-time)

<a id="q3"></a>
### Q3. Kafka Partitions for the Peak

🟢 Warm-up · *Handle bursts*

Given peak events per second, event bytes and MB/s per partition, return partitions, rounded up, with a headroom multiple.

**Examples**

- `peak_eps = 17000, size = 1000, mb_each = 10, headroom = 2` → `4`

In [ ]:
def kafka_partitions(peak_eps, size, mb_each, headroom):
    """
    Given peak events per second, event bytes and MB/s per partition, return
    partitions, rounded up, with a headroom multiple.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(kafka_partitions, [
    ((17000, 1000, 10, 2), 4),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Step by step.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Compact.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Step by step.

```python
def kafka_partitions(peak_eps, size, mb_each, headroom):
    mb = peak_eps * size / 1e6
    need = mb * headroom / mb_each
    return math.ceil(need)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Compact.

```python
def kafka_partitions(peak_eps, size, mb_each, headroom):
    return math.ceil(peak_eps * size * headroom / 1e6 / mb_each)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Step by step. That costs O(1) time and O(1) space."

"I can do better: Compact. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/system-design/kafka-partitions-for-the-peak)

<a id="q4"></a>
### Q4. Batch, Micro-Batch or Streaming?

🟢 Warm-up · *Match the need*

Given freshness needed in seconds: under 10 is 'streaming', up to 900 'micro-batch', otherwise 'batch'. Also return the design's rough complexity, 3, 2 or 1.

**Examples**

- `seconds = 5` → `('streaming', 3)`
- `seconds = 300` → `('micro-batch', 2)`
- `seconds = 86400` → `('batch', 1)`

In [ ]:
def processing_mode(seconds):
    """
    Given freshness needed in seconds: under 10 is 'streaming', up to 900
    'micro-batch', otherwise 'batch'. Also return the design's rough complexity,
    3, 2 or 1.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(processing_mode, [
    ((5,), ('streaming', 3)),
    ((300,), ('micro-batch', 2)),
    ((86400,), ('batch', 1)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Ifs.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Conditional expression.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Ifs.

```python
def processing_mode(seconds):
    if seconds < 10:
        return ("streaming", 3)
    if seconds <= 900:
        return ("micro-batch", 2)
    return ("batch", 1)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Conditional expression.

```python
def processing_mode(seconds):
    return ("streaming", 3) if seconds < 10 else ("micro-batch", 2) if seconds <= 900 else ("batch", 1)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Ifs. That costs O(1) time and O(1) space."

"I can do better: Conditional expression. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/system-design/batch-micro-batch-or-streaming)

<a id="q5"></a>
### Q5. End-to-End Latency Budget

🟢 Warm-up · *Add up the hops*

Given stage latencies in seconds, return the total, whether it meets the target, and the largest stage.

**Examples**

- `stages = {'ingest': 2, 'process': 30, 'load': 10, 'query': 1}, target = 60` → `(43, True, 'process')`

In [ ]:
def latency_budget(stages, target):
    """
    Given stage latencies in seconds, return the total, whether it meets the
    target, and the largest stage.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(latency_budget, [
    (({'ingest': 2, 'process': 30, 'load': 10, 'query': 1}, 60), (43, True, 'process')),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Built-ins.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A loop.

```python
def latency_budget(stages, target):
    total, big = 0, None
    for name, s in stages.items():
        total += s
        if big is None or s > stages[big]:
            big = name
    return (total, total <= target, big)
```

**Cost:** O(s) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Built-ins.

```python
def latency_budget(stages, target):
    t = sum(stages.values())
    return (t, t <= target, max(stages, key=stages.get))
```

**Cost:** O(s) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop. That costs O(s) time and O(1) space."

"I can do better: Built-ins. That's O(s) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/system-design/end-to-end-latency-budget)

<a id="q6"></a>
### Q6. Daily Active Users From Events

🟢 Warm-up · *A core metric*

Given (day, user) events, return distinct users per day, sorted. Reprocessing should give the same answer, however many duplicates.

**Examples**

- `events = [('d1', 'a'), ('d1', 'a'), ('d1', 'b'), ('d2', 'a')]` → `{'d1': 2, 'd2': 1}`

In [ ]:
def dau(events):
    """
    Given (day, user) events, return distinct users per day, sorted.
    Reprocessing should give the same answer, however many duplicates.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(dau, [
    (([('d1', 'a'), ('d1', 'a'), ('d1', 'b'), ('d2', 'a')],), {'d1': 2, 'd2': 1}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A dictionary of sets.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Set of pairs, then count.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A dictionary of sets.

```python
def dau(events):
    days = {}
    for day, user in events:
        days.setdefault(day, set()).add(user)
    return {d: len(days[d]) for d in sorted(days)}
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Set of pairs, then count.

```python
def dau(events):
    from collections import Counter
    return dict(sorted(Counter(d for d, _ in set(events)).items()))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A dictionary of sets. That costs O(n) time and O(n) space."

"I can do better: Set of pairs, then count. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/system-design/daily-active-users-from-events)

<a id="q7"></a>
### Q7. Fan-Out to Many Consumers

🟢 Warm-up · *Kafka vs point-to-point*

With p producers and c consumers, return direct connections needed, and connections through a central log.

**Examples**

- `p = 5, c = 10` → `(50, 15)`

In [ ]:
def connections(p, c):
    """
    With p producers and c consumers, return direct connections needed, and
    connections through a central log.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(connections, [
    ((5, 10), (50, 15)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Multiply and add.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Compact.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Multiply and add.

```python
def connections(p, c):
    direct = p * c
    via_log = p + c
    return (direct, via_log)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Compact.

```python
def connections(p, c):
    return (p * c, p + c)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Multiply and add. That costs O(1) time and O(1) space."

"I can do better: Compact. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/system-design/fan-out-to-many-consumers)

<a id="q8"></a>
### Q8. Rank the Risks

🟢 Warm-up · *Likelihood × impact*

Given risks as (name, likelihood 1-5, impact 1-5), return names ordered by score descending, ties alphabetical.

**Examples**

- `risks = [('late data', 5, 2), ('source outage', 2, 5), ('schema change', 4, 4)]` → `['schema change', 'late data', 'source outage']`

In [ ]:
def rank_risks(risks):
    """
    Given risks as (name, likelihood 1-5, impact 1-5), return names ordered by
    score descending, ties alphabetical.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(rank_risks, [
    (([('late data', 5, 2), ('source outage', 2, 5), ('schema change', 4, 4)],), ['schema change', 'late data', 'source outage']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compute, then sort pairs.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A sort key.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Compute, then sort pairs.

```python
def rank_risks(risks):
    scored = []
    for name, l, i in risks:
        scored.append((-l * i, name))
    scored.sort()
    return [n for _, n in scored]
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A sort key.

```python
def rank_risks(risks):
    return [r[0] for r in sorted(risks, key=lambda r: (-r[1] * r[2], r[0]))]
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compute, then sort pairs. That costs O(n log n) time and O(n) space."

"I can do better: A sort key. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/system-design/rank-the-risks)

<a id="q9"></a>
### Q9. Growth Headroom

🟢 Warm-up · *When will it break?*

Data grows by a monthly rate. Given current load, capacity and growth rate, return whole months until load exceeds capacity.

**Examples**

- `load = 100, capacity = 400, rate = 0.1` → `15`
- `load = 100, capacity = 100, rate = 0.1` → `1`

In [ ]:
def months_left(load, capacity, rate):
    """
    Data grows by a monthly rate. Given current load, capacity and growth rate,
    return whole months until load exceeds capacity.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(months_left, [
    ((100, 400, 0.1), 15),
    ((100, 100, 0.1), 1),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compound in a loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A logarithm, with care for exact boundaries.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Compound in a loop.

```python
def months_left(load, capacity, rate):
    months = 0
    while load <= capacity:
        load *= 1 + rate
        months += 1
    return months
```

**Cost:** O(m) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A logarithm, with care for exact boundaries.

```python
def months_left(load, capacity, rate):
    return math.floor(math.log(capacity / load) / math.log(1 + rate)) + 1
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compound in a loop. That costs O(m) time and O(1) space."

"I can do better: A logarithm, with care for exact boundaries. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/system-design/growth-headroom)

<a id="q10"></a>
### Q10. Will the Batch Finish in Time?

🟡 Interview standard · *Check the window*

A job must process gb within window hours. With MB/s per core and an efficiency share, return cores needed, rounded up.

**Examples**

- `gb = 5000, window = 2, mb_core = 20, efficiency = 0.7` → `50`

In [ ]:
def cores_needed(gb, window, mb_core, efficiency):
    """
    A job must process gb within window hours. With MB/s per core and an
    efficiency share, return cores needed, rounded up.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(cores_needed, [
    ((5000, 2, 20, 0.7), 50),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Convert units step by step.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

One formula.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Convert units step by step.

```python
def cores_needed(gb, window, mb_core, efficiency):
    mb = gb * 1000
    seconds = window * 3600
    rate = mb / seconds
    return math.ceil(rate / (mb_core * efficiency))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** One formula.

```python
def cores_needed(gb, window, mb_core, efficiency):
    return math.ceil(gb * 1000 / (window * 3600) / (mb_core * efficiency))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Convert units step by step. That costs O(1) time and O(1) space."

"I can do better: One formula. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/system-design/will-the-batch-finish-in-time)

<a id="q11"></a>
### Q11. Pick a Serving Store

🟡 Interview standard · *It depends on the query*

Given latency in ms and whether queries are by key: key lookups under 50 ms → 'key-value store'; aggregates under 1000 ms → 'real-time OLAP'; otherwise 'warehouse'.

**Examples**

- `latency_ms = 10, by_key = True` → `'key-value store'`
- `latency_ms = 500, by_key = False` → `'real-time OLAP'`
- `latency_ms = 30000, by_key = False` → `'warehouse'`

In [ ]:
def serving_store(latency_ms, by_key):
    """
    Given latency in ms and whether queries are by key: key lookups under 50 ms
    → 'key-value store'; aggregates under 1000 ms → 'real-time OLAP'; otherwise
    'warehouse'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(serving_store, [
    ((10, True), 'key-value store'),
    ((500, False), 'real-time OLAP'),
    ((30000, False), 'warehouse'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Ifs.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Conditional expression.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Ifs.

```python
def serving_store(latency_ms, by_key):
    if by_key and latency_ms < 50:
        return "key-value store"
    if latency_ms < 1000:
        return "real-time OLAP"
    return "warehouse"
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Conditional expression.

```python
def serving_store(latency_ms, by_key):
    return "key-value store" if by_key and latency_ms < 50 else "real-time OLAP" if latency_ms < 1000 else "warehouse"
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Ifs. That costs O(1) time and O(1) space."

"I can do better: Conditional expression. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/system-design/pick-a-serving-store)

<a id="q12"></a>
### Q12. Clickstream: Bot Filter

🟡 Interview standard · *Clean before counting*

Drop users with more than limit events in the data. Return (kept events, bots sorted).

**Examples**

- `events = [('a', 1), ('b', 1), ('b', 2), ('b', 3), ('c', 1)], limit = 2` → `([('a', 1), ('c', 1)], ['b'])`

In [ ]:
def bot_filter(events, limit):
    """
    Drop users with more than limit events in the data. Return (kept events,
    bots sorted).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(bot_filter, [
    (([('a', 1), ('b', 1), ('b', 2), ('b', 3), ('c', 1)], 2), ([('a', 1), ('c', 1)], ['b'])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A dictionary, then filter.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`Counter`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A dictionary, then filter.

```python
def bot_filter(events, limit):
    counts = {}
    for u, _ in events:
        counts[u] = counts.get(u, 0) + 1
    bots = sorted(u for u, c in counts.items() if c > limit)
    return ([e for e in events if e[0] not in bots], bots)
```

**Cost:** O(n) time, O(u) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `Counter`.

```python
def bot_filter(events, limit):
    from collections import Counter
    c = Counter(u for u, _ in events)
    bots = {u for u, n in c.items() if n > limit}
    return ([e for e in events if e[0] not in bots], sorted(bots))
```

**Cost:** O(n) time, O(u) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A dictionary, then filter. That costs O(n) time and O(u) space."

"I can do better: `Counter`. That's O(n) time and O(u) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/system-design/clickstream-bot-filter)

<a id="q13"></a>
### Q13. CDC Into a Lake: Plan the Merge

🟡 Interview standard · *Compact, then merge*

Given a target, and CDC events (seq, op, id, value) possibly out of order, keep the highest seq per id, then apply. Return the target sorted.

**Examples**

- `target = {1: 'a'}, events = [(2, 'u', 1, 'B'), (1, 'u', 1, 'OLD'), (3, 'c', 2, 'x'), (4, 'd', 2, None)]` → `{1: 'B'}`

In [ ]:
def cdc_merge(target, events):
    """
    Given a target, and CDC events (seq, op, id, value) possibly out of order,
    keep the highest seq per id, then apply. Return the target sorted.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(cdc_merge, [
    (({1: 'a'}, [(2, 'u', 1, 'B'), (1, 'u', 1, 'OLD'), (3, 'c', 2, 'x'), (4, 'd', 2, None)]), {1: 'B'}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Sort by seq, then replay all.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A dictionary of the highest seq per id, then one pass.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Sort by seq, then replay all.

```python
def cdc_merge(target, events):
    out = dict(target)
    for seq, op, key, value in sorted(events):
        if op == "d":
            out.pop(key, None)
        else:
            out[key] = value
    return dict(sorted(out.items()))
```

**Cost:** O(n log n) time, O(t) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A dictionary of the highest seq per id, then one pass.

```python
def cdc_merge(target, events):
    latest = {}
    for e in events:
        if e[2] not in latest or e[0] > latest[e[2]][0]:
            latest[e[2]] = e
    out = dict(target)
    for _, op, key, value in latest.values():
        if op == "d":
            out.pop(key, None)
        else:
            out[key] = value
    return dict(sorted(out.items()))
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Sort by seq, then replay all. That costs O(n log n) time and O(t) space."

"I can do better: A dictionary of the highest seq per id, then one pass. That's O(n) time and O(k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/system-design/cdc-into-a-lake-plan-the-merge)

<a id="q14"></a>
### Q14. Real-Time Counter With Rollups

🟡 Interview standard · *Fast dashboards*

Keep per-minute counts, and roll them up to hours for faster reads. Given (minute, n) rows, return {hour: total}, sorted.

**Examples**

- `rows = [(0, 5), (59, 5), (60, 1), (125, 2)]` → `{0: 10, 1: 1, 2: 2}`

In [ ]:
def rollup(rows):
    """
    Keep per-minute counts, and roll them up to hours for faster reads. Given
    (minute, n) rows, return {hour: total}, sorted.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(rollup, [
    (([(0, 5), (59, 5), (60, 1), (125, 2)],), {0: 10, 1: 1, 2: 2}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A dictionary.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`defaultdict`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A dictionary.

```python
def rollup(rows):
    out = {}
    for minute, n in rows:
        h = minute // 60
        out[h] = out.get(h, 0) + n
    return dict(sorted(out.items()))
```

**Cost:** O(n) time, O(h) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `defaultdict`.

```python
def rollup(rows):
    from collections import defaultdict
    out = defaultdict(int)
    for m, n in rows:
        out[m // 60] += n
    return dict(sorted(out.items()))
```

**Cost:** O(n) time, O(h) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A dictionary. That costs O(n) time and O(h) space."

"I can do better: `defaultdict`. That's O(n) time and O(h) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/system-design/real-time-counter-with-rollups)

<a id="q15"></a>
### Q15. Tenant Quotas

🟡 Interview standard · *One customer shouldn't starve the rest*

Share capacity fairly: each tenant gets up to its demand, and leftovers go to those still wanting more (max-min fairness). Return allocations sorted by name.

**Examples**

- `demand = {'a': 10, 'b': 50, 'c': 100}, capacity = 90` → `{'a': 10, 'b': 40.0, 'c': 40.0}`

In [ ]:
def fair_share(demand, capacity):
    """
    Share capacity fairly: each tenant gets up to its demand, and leftovers go
    to those still wanting more (max-min fairness). Return allocations sorted by
    name.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(fair_share, [
    (({'a': 10, 'b': 50, 'c': 100}, 90), {'a': 10, 'b': 40.0, 'c': 40.0}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Repeatedly give an equal share.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Sort by demand, then fill smallest first.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Repeatedly give an equal share.

```python
def fair_share(demand, capacity):
    alloc = {t: 0 for t in demand}
    left = capacity
    active = [t for t in demand if demand[t] > 0]
    while left > 1e-9 and active:
        share = left / len(active)
        for t in list(active):
            give = min(share, demand[t] - alloc[t])
            alloc[t] += give
            left -= give
            if demand[t] - alloc[t] <= 1e-9:
                active.remove(t)
    return {t: round(alloc[t], 2) for t in sorted(alloc)}
```

**Cost:** O(t²) time, O(t) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Sort by demand, then fill smallest first.

```python
def fair_share(demand, capacity):
    out, left = {}, capacity
    order = sorted(demand, key=demand.get)
    for i, t in enumerate(order):
        out[t] = round(min(demand[t], left / (len(order) - i)), 2)
        left -= out[t]
    return dict(sorted(out.items()))
```

**Cost:** O(t log t) time, O(t) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Repeatedly give an equal share. That costs O(t²) time and O(t) space."

"I can do better: Sort by demand, then fill smallest first. That's O(t log t) time and O(t) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/system-design/tenant-quotas)

<a id="q16"></a>
### Q16. How Long Can You Replay?

🟡 Interview standard · *Retention vs recovery*

After an outage of h hours, can you recover from Kafka? Return (can recover, hours to catch up at speed × input, rounded to 2 decimals).

**Examples**

- `outage = 6, retention = 168, speed = 3` → `(True, 3.0)`
- `outage = 200, retention = 168, speed = 3` → `(False, 100.0)`

In [ ]:
def can_replay(outage, retention, speed):
    """
    After an outage of h hours, can you recover from Kafka? Return (can recover,
    hours to catch up at speed × input, rounded to 2 decimals).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(can_replay, [
    ((6, 168, 3), (True, 3.0)),
    ((200, 168, 3), (False, 100.0)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Ifs and a division.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Compact.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Ifs and a division.

```python
def can_replay(outage, retention, speed):
    ok = outage <= retention
    hours = outage / (speed - 1)
    return (ok, round(hours, 2))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Compact.

```python
def can_replay(outage, retention, speed):
    return (outage <= retention, round(outage / (speed - 1), 2))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Ifs and a division. That costs O(1) time and O(1) space."

"I can do better: Compact. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/system-design/how-long-can-you-replay)

<a id="q17"></a>
### Q17. Monthly Cost Estimate

🟡 Interview standard · *Every design has a price*

With made-up unit prices, return the monthly total: storage\_tb × 23, compute\_hours × 0.5 × nodes, streaming\_mb × 0.04 × 730. Round to 2 decimals, and return the largest part.

**Examples**

- `storage_tb = 100, hours = 300, nodes = 10, streaming_mb = 50` → `(5260.0, 'storage')`

In [ ]:
def monthly_cost(storage_tb, hours, nodes, streaming_mb):
    """
    With made-up unit prices, return the monthly total: storage_tb × 23,
    compute_hours × 0.5 × nodes, streaming_mb × 0.04 × 730. Round to 2 decimals,
    and return the largest part.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(monthly_cost, [
    ((100, 300, 10, 50), (5260.0, 'storage')),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compute each.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A dictionary.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Compute each.

```python
def monthly_cost(storage_tb, hours, nodes, streaming_mb):
    storage = storage_tb * 23
    compute = hours * 0.5 * nodes
    stream = streaming_mb * 0.04 * 730
    total = storage + compute + stream
    if storage >= compute and storage >= stream:
        big = "storage"
    elif compute >= stream:
        big = "compute"
    else:
        big = "streaming"
    return (round(total, 2), big)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A dictionary.

```python
def monthly_cost(storage_tb, hours, nodes, streaming_mb):
    parts = {"storage": storage_tb * 23, "compute": hours * 0.5 * nodes, "streaming": streaming_mb * 0.04 * 730}
    return (round(sum(parts.values()), 2), max(parts, key=parts.get))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compute each. That costs O(1) time and O(1) space."

"I can do better: A dictionary. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/system-design/monthly-cost-estimate)

<a id="q18"></a>
### Q18. Design a Partition Layout

🟡 Interview standard · *Year, month, day or hour?*

Choose the coarsest of 'month', 'day' or 'hour' so each partition stays at or below max\_gb. Given GB per day, return the choice and its GB, rounded to 2 decimals. If even an hour's too big, return 'hour' anyway.

**Examples**

- `gb_day = 10, max_gb = 500` → `('month', 300)`
- `gb_day = 500, max_gb = 100` → `('hour', 20.83)`
- `gb_day = 5000, max_gb = 100` → `('hour', 208.33)`

In [ ]:
def partition_grain(gb_day, max_gb):
    """
    Choose the coarsest of 'month', 'day' or 'hour' so each partition stays at
    or below max_gb. Given GB per day, return the choice and its GB, rounded to
    2 decimals. If even an hour's too big, return 'hour' anyway.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(partition_grain, [
    ((10, 500), ('month', 300)),
    ((500, 100), ('hour', 20.83)),
    ((5000, 100), ('hour', 208.33)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Ifs.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A loop over options.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Ifs.

```python
def partition_grain(gb_day, max_gb):
    if gb_day * 30 <= max_gb:
        return ("month", round(gb_day * 30, 2))
    if gb_day <= max_gb:
        return ("day", round(gb_day, 2))
    return ("hour", round(gb_day / 24, 2))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A loop over options.

```python
def partition_grain(gb_day, max_gb):
    for name, factor in (("month", 30), ("day", 1)):
        if gb_day * factor <= max_gb:
            return (name, round(gb_day * factor, 2))
    return ("hour", round(gb_day / 24, 2))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Ifs. That costs O(1) time and O(1) space."

"I can do better: A loop over options. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/system-design/design-a-partition-layout)

<a id="q19"></a>
### Q19. Sharding a Key-Value Table

🟡 Interview standard · *Scale reads and writes*

Return shards needed for read and write rates, and for data size, taking the largest and rounding up. Each shard handles reads\_each, writes\_each and gb\_each.

**Examples**

- `reads = 100000, writes = 10000, gb = 2000, reads_each = 10000, writes_each = 2000, gb_each = 500` → `10`

In [ ]:
def shards(reads, writes, gb, reads_each, writes_each, gb_each):
    """
    Return shards needed for read and write rates, and for data size, taking the
    largest and rounding up. Each shard handles reads_each, writes_each and
    gb_each.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(shards, [
    ((100000, 10000, 2000, 10000, 2000, 500), 10),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Three divisions.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`max` of them.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Three divisions.

```python
def shards(reads, writes, gb, reads_each, writes_each, gb_each):
    a = math.ceil(reads / reads_each)
    b = math.ceil(writes / writes_each)
    c = math.ceil(gb / gb_each)
    return max(a, b, c)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `max` of them.

```python
def shards(reads, writes, gb, reads_each, writes_each, gb_each):
    return math.ceil(max(reads / reads_each, writes / writes_each, gb / gb_each))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Three divisions. That costs O(1) time and O(1) space."

"I can do better: `max` of them. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/system-design/sharding-a-key-value-table)

<a id="q20"></a>
### Q20. Data Mesh Ownership

🟡 Interview standard · *Who owns which data?*

Given datasets with owners, return datasets with no owner, and owners with more than limit datasets (a sign that one team's a bottleneck), both sorted.

**Examples**

- `datasets = {'a': 'x', 'b': 'x', 'c': 'x', 'd': None, 'e': 'y'}, limit = 2` → `(['d'], ['x'])`

In [ ]:
def ownership(datasets, limit):
    """
    Given datasets with owners, return datasets with no owner, and owners with
    more than limit datasets (a sign that one team's a bottleneck), both sorted.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(ownership, [
    (({'a': 'x', 'b': 'x', 'c': 'x', 'd': None, 'e': 'y'}, 2), (['d'], ['x'])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loops.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`Counter`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Loops.

```python
def ownership(datasets, limit):
    orphans, counts = [], {}
    for name, owner in datasets.items():
        if owner is None:
            orphans.append(name)
        else:
            counts[owner] = counts.get(owner, 0) + 1
    return (sorted(orphans), sorted(o for o, c in counts.items() if c > limit))
```

**Cost:** O(d) time, O(o) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `Counter`.

```python
def ownership(datasets, limit):
    from collections import Counter
    c = Counter(o for o in datasets.values() if o)
    return (sorted(n for n, o in datasets.items() if o is None), sorted(o for o, n in c.items() if n > limit))
```

**Cost:** O(d) time, O(o) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loops. That costs O(d) time and O(o) space."

"I can do better: `Counter`. That's O(d) time and O(o) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/system-design/data-mesh-ownership)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. How do you choose between Lambda and Kappa?</b></summary>

Kappa is simpler if one stream path can do everything, and replay is feasible. Lambda may suit heavy historical computation alongside real-time needs, at the cost of two code paths.

</details>

<details>
<summary><b>2. How do you handle a sudden 10× traffic spike?</b></summary>

Kafka buffers it, consumers autoscale, and backpressure limits intake. Dashboards may lag briefly but no data's lost, as long as retention covers the catch-up.

</details>

<details>
<summary><b>3. Warehouse or lakehouse?</b></summary>

A warehouse is simplest for SQL analytics. A lakehouse suits very large or mixed workloads, machine learning, and using several engines on open formats.

</details>

<details>
<summary><b>4. How do you guarantee no data loss?</b></summary>

Durable ingest with replication, acks=all, keeping raw data, checkpoints, reconciliation, and alerts when counts differ.

</details>

<details>
<summary><b>5. How do you make a design cheaper?</b></summary>

Reduce freshness where it isn't needed, use columnar formats and partitions, tier old data, autoscale, use spot instances for batch, and remove unused tables.

</details>

<details>
<summary><b>6. Scenario: design a daily sales report for 1,000 stores.</b></summary>

Probably batch: extract nightly or via CDC, load raw, build a star schema with dbt, add quality checks and reconciliation against totals, then publish atomically. Avoid streaming unless needed.

</details>

<details>
<summary><b>7. Scenario: you must add real-time fraud features to a batch platform.</b></summary>

Add a stream from Kafka for velocity features, stored in a low-latency store. Share definitions with batch to avoid skew, and keep batch as the source of truth for training.

</details>

<details>
<summary><b>8. Scenario: the interviewer pushes back on your choice. How do you respond?</b></summary>

Treat it as useful information. Ask what constraint they're thinking of, explain your trade-off again, and change your design if the new information justifies it.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: How to approach any design question
- [ ] I can explain: Walkthrough: clickstream analytics
- [ ] I can explain: Walkthrough: replicate a database to the lake
- [ ] I can explain: Scale, cost and the human side

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) How Long Can You Replay?
- [ ] [Q17](#q17) Monthly Cost Estimate
- [ ] [Q18](#q18) Design a Partition Layout
- [ ] [Q19](#q19) Sharding a Key-Value Table
- [ ] [Q20](#q20) Data Mesh Ownership

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.